# 02 — Data cleaning

Standardise each raw StatCan table into a tidy, snake_case frame; parse dates;
harmonise geography; and **flag missing/suppressed values without inventing them**.
Cleaned files are written to `data/processed/`.

Cleaning helpers live in `src/data_cleaning.py`.

> **Governance reminder.** This dashboard produces a *triage* signal for human policy review. It does **not** determine social-assistance need, eligibility, or benefits. Claude outputs are claims, not facts. Missing or suppressed values are flagged, never invented. Any policy interpretation is reviewed by the AI Council before it is treated as decision-ready.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))  # import project modules
import utils
print("Project root:", utils.ROOT)
print("Raw data:    ", utils.RAW)
print("Processed:   ", utils.PROCESSED)
print("Outputs:     ", utils.OUTPUTS)
utils.ensure_dirs()

In [ ]:
import pandas as pd, numpy as np
import data_cleaning as dc
from datetime import date

## Cleaning rules applied

- Column names → lowercase snake_case.
- `REF_DATE` → `ref_date` + derived `year`, `month`, `quarter`, `data_frequency`.
- `GEO` → `geo`, `DGUID` → `dguid`, `VALUE` → `value` (numeric).
- Suppressed / blank / `F` / `..` → `NaN` **and** `missing_value_flag = True`.
- Provenance columns attached: `source_product_id`, `source_table_title`, `source_category`, `retrieval_date`.

## Labour Force Survey (the spine)

In [ ]:
lfs_raw = pd.read_csv(utils.RAW / "labour_force" / "lfs_14100287_tidy.csv", low_memory=False)
print(f"raw rows: {len(lfs_raw):,}")
lfs_clean = dc.clean_statcan(
    lfs_raw, product_id=14100287, frequency="monthly",
    table_title="Labour force characteristics, monthly (LFS), 14-10-0287-01",
    category="labour force", retrieval_date=date.today().isoformat())

# Keep ESTIMATE rows (drop the standard-error rows) for the analysis-ready file
stat = dc.find_col(lfs_clean, "statistics", "statistic")
lfs_est = lfs_clean[lfs_clean[stat].astype(str).str.fullmatch("Estimate")]
miss = int(lfs_est["missing_value_flag"].sum())
print(f"estimate rows: {len(lfs_est):,} | missing/suppressed: {miss:,} ({miss/len(lfs_est):.1%})")

In [ ]:
keep = ["ref_date","year","month","quarter","data_frequency","geo","dguid",
        "labour_force_characteristics","gender","age_group","statistics","data_type",
        "uom","value","status","missing_value_flag",
        "source_product_id","source_table_title","source_category","retrieval_date"]
keep = [c for c in keep if c in lfs_est.columns]
lfs_est[keep].to_csv(utils.PROCESSED / "labour_force_clean.csv", index=False)
print("wrote data/processed/labour_force_clean.csv")
lfs_est[keep].head()

## Income / demographics / affordability tables

Run these once the raw tables are downloaded (notebook 01). The same `clean_statcan()` standardiser applies. Each writes a `*_clean.csv`.

In [ ]:
def clean_if_present(pid, domain, fname, frequency, title, category):
    src = utils.RAW / domain / f"{pid}.csv"
    if not src.exists():
        print(f"skip {fname} — {src} not found (run notebook 01)"); return None
    raw = pd.read_csv(src, low_memory=False)
    out = dc.clean_statcan(raw, product_id=pid, frequency=frequency,
                           table_title=title, category=category,
                           retrieval_date=date.today().isoformat())
    out.to_csv(utils.PROCESSED / fname, index=False)
    print(f"wrote data/processed/{fname}  ({len(out):,} rows)")
    return out

# Income (combine the income-domain tables into income_clean.csv when present)
clean_if_present(9810059701, "income", "income_census_2021_clean.csv", "one-time",
                 "Employment income (2021 Census), 98-10-0597-01", "income")
clean_if_present(1410041701, "income", "wages_occupation_annual_clean.csv", "annual",
                 "Employee wages by occupation, annual, 14-10-0417-01", "income/wages")
clean_if_present(1410042601, "income", "wages_occupation_monthly_clean.csv", "monthly",
                 "Employee wages by occupation, monthly, 14-10-0426-01", "income/wages")
clean_if_present(17100005, "demographics", "demographics_clean.csv", "annual",
                 "Population estimates, 17-10-0005-01", "demographics")
clean_if_present(18100004, "affordability", "affordability_clean.csv", "monthly",
                 "CPI incl. Shelter (proxy), 18-10-0004-01", "affordability")

> **Frequency mismatch note.** The LFS spine is monthly; income/wages/Census/population are annual or one-time. Do not force them together at monthly grain — they are joined by `year` as labelled annual *context* in notebook 05. See `data/metadata/integration_notes.md`.